# Hallucination Detection in LLMs via Dependency Graph Matching
**CS221 – Natural Language Processing**

This notebook provides interactive exploration of:
1. XSum dataset samples
2. Generated summaries from local LLM
3. Dependency graph construction
4. SVO triple extraction
5. Graph matching hallucination scores
6. Evaluation results & error analysis

In [ ]:
import sys
sys.path.insert(0, '..')  # project root

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from src.utils import load_config, load_json
from src.data.loader import load_xsum
from src.graph import DependencyParser, GraphBuilder, GraphMatcher

cfg = load_config('../config/config.yaml')
print('Config loaded.')

## 1. Explore XSum Dataset

In [ ]:
# Load a small slice for exploration
samples = load_xsum(split='test', num_samples=20)
pd.DataFrame([
    {'id': s['id'], 'doc_len': len(s['document'].split()),
     'summary': s['summary_ref'][:80], 'label': s['label']}
    for s in samples
])

## 2. Dependency Parsing — Single Example

In [ ]:
parser = DependencyParser('en_core_web_sm')

text = samples[0]['summary_ref']
parsed = parser.parse(text)

print('Text:', text)
print('\nNamed Entities:', parsed.entities)
print('\nSVO Triples:')
for t in parsed.svo_triples:
    print(f'  ({t.subject}, {t.verb}, {t.obj})')
print('\nDep Triples (first 10):')
for t in parsed.dep_triples[:10]:
    print(f'  {t.head} --[{t.relation}]--> {t.child}')

## 3. Dependency Graph Visualisation

In [ ]:
builder = GraphBuilder(include_ner=True, include_svo=True)

s = samples[0]
doc_parsed = parser.parse(s['document'])
sum_parsed = parser.parse(s['summary_ref'])

G_doc = builder.build(doc_parsed)
G_sum = builder.build(sum_parsed)

print(f'Document graph: {G_doc.number_of_nodes()} nodes, {G_doc.number_of_edges()} edges')
print(f'Summary graph : {G_sum.number_of_nodes()} nodes, {G_sum.number_of_edges()} edges')

In [ ]:
def plot_graph(G, title, max_nodes=35, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    if G.number_of_nodes() > max_nodes:
        top = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:max_nodes]
        G = G.subgraph([n for n, _ in top]).copy()
    
    pos = nx.spring_layout(G, seed=42, k=1.8)
    colors = ['#4A90D9' if d.get('type') == 'WORD' else '#E67E22'
              for _, d in G.nodes(data=True)]
    nx.draw(G, pos, ax=ax, with_labels=True, node_color=colors,
            node_size=500, font_size=7, font_color='white',
            arrows=True, arrowsize=12)
    ax.set_title(title, fontweight='bold')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
plot_graph(G_doc, 'Source Document Graph', ax=axes[0])
plot_graph(G_sum, 'Reference Summary Graph', ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Graph Matching — Manual Example

In [ ]:
matcher = GraphMatcher(
    svo_weight=0.40, entity_weight=0.35, lexical_weight=0.25,
    threshold=0.50, use_fuzzy_match=True
)

# Compare document against its own reference summary → should be faithful
result = matcher.match(doc_parsed, sum_parsed, G_doc, G_sum)
print(json.dumps(result.to_dict(), indent=2))

print('\nMatched SVO triples  :', result.matched_triples)
print('Unmatched SVO triples:', result.unmatched_triples)

## 5. Load & Analyse Full Detection Results

In [ ]:
# Load saved detection results
try:
    detections = load_json('../outputs/results/detections.json')
    df = pd.DataFrame([
        {
            'id': s['id'],
            'label': s.get('label', -1),
            'predicted': s.get('predicted_label', -1),
            **s.get('detection', {}),
        }
        for s in detections
    ])
    print(f'Loaded {len(df)} detection results.')
    df.head()
except FileNotFoundError:
    print('Run main.py first to generate detections.')

In [ ]:
# Score distribution by label
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
score_cols = ['svo_score', 'entity_score', 'lexical_score']

for ax, col in zip(axes, score_cols):
    if col not in df.columns:
        continue
    for lbl, grp in df.groupby('label'):
        grp[col].hist(ax=ax, alpha=0.6, bins=20,
                      label='Faithful' if lbl == 0 else 'Hallucinated')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('Score')
    ax.legend()

plt.suptitle('Score Distributions by True Label', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of signals
score_df = df[['svo_score', 'entity_score', 'lexical_score',
               'composite_score', 'hallucination_score']].dropna()
plt.figure(figsize=(7, 5))
sns.heatmap(score_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Signal Correlation Matrix')
plt.tight_layout()
plt.show()

## 6. Error Analysis

In [ ]:
# False positives and false negatives
annotated = df[df['label'] != -1].copy()

fp = annotated[(annotated['label'] == 0) & (annotated['predicted'] == 1)]
fn = annotated[(annotated['label'] == 1) & (annotated['predicted'] == 0)]

print(f'False Positives (faithful → predicted hallucinated): {len(fp)}')
print(f'False Negatives (hallucinated → predicted faithful): {len(fn)}')

# Show worst FP
print('\nTop False Positives (highest hallucination score among faithful):')
fp_details = []
for _, row in fp.nlargest(5, 'hallucination_score').iterrows():
    s = next(x for x in detections if x['id'] == row['id'])
    fp_details.append({
        'id': row['id'],
        'hal_score': round(row['hallucination_score'], 3),
        'summary_gen': s.get('summary_gen', '')[:80],
        'summary_ref': s.get('summary_ref', '')[:80],
    })
pd.DataFrame(fp_details)

In [ ]:
# Ablation results (if available)
try:
    ablation = pd.read_csv('../outputs/results/ablation.csv')
    print('Top 10 weight combinations by F1:')
    display(ablation.head(10))

    fig, ax = plt.subplots(figsize=(8, 5))
    scatter = ax.scatter(
        ablation['svo_weight'], ablation['entity_weight'],
        c=ablation['f1'], cmap='viridis', s=80, alpha=0.8
    )
    plt.colorbar(scatter, label='F1')
    ax.set_xlabel('SVO Weight')
    ax.set_ylabel('Entity Weight')
    ax.set_title('F1 by Weight Configuration')
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print('Run scripts/ablation_study.py first.')